**Avvio locale e Colab:** preparare la cartella completa seguendo il [README del laboratorio](../README.md). Conservare gli identificativi delle celle e salvare gli output nel proprio runtime.

<!-- C01 -->
# Transfer Learning Studio
Computer Vision · Lezione 14

### Elaborato individuale — da compilare


Il notebook della lezione 13 è una dimostrazione completa da consultare. Qui sono richieste **cinque attività guidate**: i tre completamenti di codice **C11, C19, C29** e i richiami testuali **P01 in C13**, **P02 in C17**. P01 e P02 non richiedono nuovo codice o training. Conservare i **39 ID C01–C39** e le 21 celle di codice.

Prevedere **4–6 ore complessive nella settimana**, di cui circa **20** per P01/P02: è una stima organizzativa, non un tempo di calcolo misurato. Eseguire soltanto gli stadi già previsti: LP di 20 epoche e FT parziale di 3 per ciascuno dei due modelli; non aggiungere corse, seed, download di dati/pesi o valutazioni del test. Il piccolo esempio residuale è analitico, non un nuovo esperimento.

Rispondere a **N01–N06 in 3–5 righe o una piccola tabella**, richiamando gli output senza ricopiarli. N06 è una breve lettura della diagnosi già fornita in C34, non una challenge da implementare. Scrivere **F01–F05 direttamente in C36, entro 500 parole complessive**; non serve una relazione separata. P01/P02 e le risposte N si citano nella sintesi senza svolgerli una seconda volta. Gli esercizi D01–D08, la discussione conclusiva e le proposte di repliche o nuovi esperimenti sono **autostudio facoltativo, non valutato**.

Consegnare il `.ipynb` con nome, cognome, completamenti, risposte e output, insieme alla cartella `output_lezione14/` compressa. Nome suggerito: `ComputerVision_Lezione14.ipynb`. Se si usano risultati di recupero, distinguere precisamente le operazioni svolte personalmente e le misure fornite nell’archivio originale.

Confronto di ResNet18 e DeiT Tiny su Digits. Ogni cella è identificata da **Cxx**, contando anche il testo. Gli output di riferimento documentano un’esecuzione reale, i tempi dipendono dal computer.

Nei materiali del corso si usa il **punto decimale** (0.5) e la **virgola seguita da uno spazio** fra componenti di vettori e argomenti, per esempio (1.2, 1, −0.3). I conteggi interi non usano separatori delle migliaia.

<!-- C02 -->
## Protocollo fissato prima della corsa

### Lessico prima delle formule

Il **pretraining** è l’addestramento precedente su un problema **sorgente**, qui ImageNet; il problema **target** è Digits. Il **backbone** è il corpo della rete che trasforma un’immagine in una rappresentazione, o vettore di **feature** h. La **testa** è l’ultimo classificatore: produce gli **score s**, detti **logits**, per le dieci classi target. Gli score non sono ancora probabilità. Nel **linear probe (LP)** apprendono soltanto pesi e bias della nuova testa; nel **fine-tuning (FT) parziale** apprendono anche i parametri di un suffisso del backbone. Il prefisso iniziale resta fisso. Il full FT renderebbe trainabile tutto il backbone, ma non viene eseguito qui.

Un **batch** raggruppa gli esempi di un forward; uno **step** è un aggiornamento dell’ottimizzatore dopo il backward; un’**epoca** attraversa l’intero training. Un **checkpoint del modello** conserva una copia indipendente dello stato scelto, inclusi i buffer. I checkpoint didattici della percorso sono invece controlli di comprensione. Un **report** è la relazione di protocollo, risultati e limiti.

Nessun tuning sullo stress. 1077 immagini training, 360 validation, test escluso. Input 224×224 RGB replicato, nessun crop. Linear probe 20 epoche, fine-tuning **parziale** 3 epoche. Selezione separata per stadio: prima massima accuracy validation pulita. Ogni FT parte dal checkpoint LP della propria architettura. Le scelte sono didattiche e non ottimizzate per ciascun modello.

In [ ]:
# C03
from pathlib import Path
import sys, json, time, copy
# Avvio Colab nello stesso notebook: non aggiunge né rinumera celle Cxx.
import os, importlib.util, importlib.metadata, subprocess, zipfile, io
IN_COLAB=importlib.util.find_spec("google") is not None and importlib.util.find_spec("google.colab") is not None
if IN_COLAB:
    from google.colab import files
    _root=Path("/content/ComputerVision_Lezione14")
    _package=_root/"Transfer_Learning_Studio_studenti"
    if not (_package/"supporto/transfer_utils.py").exists():
        print("Carica Transfer_Learning_Studio_CV2026_studenti.zip, senza estrarlo sul browser.")
        _uploaded=files.upload()
        _archives=[v for k,v in _uploaded.items() if k.endswith(".zip")]
        if len(_archives)!=1: raise RuntimeError("Caricare un solo ZIP della Lezione 14.")
        _root.mkdir(parents=True,exist_ok=True)
        with zipfile.ZipFile(io.BytesIO(_archives[0])) as _z:
            for _entry in _z.infolist():
                if not (_root/_entry.filename).resolve().is_relative_to(_root.resolve()):
                    raise RuntimeError("Percorso non valido nello ZIP.")
            _z.extractall(_root)
        if not (_package/"supporto/transfer_utils.py").exists():
            raise RuntimeError("Lo ZIP caricato non è il pacchetto studenti della Lezione 14.")
    # Conserva la coppia torch/torchvision del runtime, non installa requirements.txt.
    import torch, torchvision
    _constraint=_root/"vincoli_colab.txt"
    _constraint.write_text(f"torch=={torch.__version__}\ntorchvision=={torchvision.__version__}\n")
    if importlib.util.find_spec("timm") is None:
        subprocess.check_call([sys.executable,"-m","pip","install","--quiet","--constraint",str(_constraint),"timm==1.0.30"])
    os.chdir(_package/"studenti")
    print("Colab: protocollo CPU; torch/torchvision conservati; directory:",Path.cwd())
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
BASE=Path.cwd().parent
if not (BASE/'supporto').exists():
    raise RuntimeError('Aprire il notebook dalla cartella studenti o docente del pacchetto estratto.')
sys.path.insert(0,str(BASE/'supporto'))
import transfer_utils as u
from transfer_utils import head_of
ENV=u.setup()
OUT=Path.cwd()/'output_lezione14';OUT.mkdir(exist_ok=True)
print(ENV)
print(u.CONFIG)

<!-- C04 -->
### Organizzazione locale
La cartella `pesi/` contiene i checkpoint ufficiali, `dati/` lo split identico alla lezione 12. Non serve rete durante lo studio. Non eseguire il notebook dentro lo ZIP. `recupero/` permette la lettura dei risultati e il ripristino dei pesi selezionati. Le verifiche dei file precedono il fit.

**Colab:** caricare questo notebook tramite File → Carica notebook. Selezionare CPU (nessun acceleratore); C03 chiede lo ZIP studenti la prima volta e prepara i percorsi nello stesso runtime. Non installare il requirements locale su Colab. Le versioni e i tempi del nuovo ambiente vanno dichiarati. Prima di chiudere scaricare output e notebook: vedere `Avvio_Colab.md`.

Il . Nello studio si leggono gli output preparati; i tre TODO C11/C19/C29 si risolvono nella propria copia. La lettura dei risultati di recupero non sostituisce i completamenti o le risposte P01/P02.

In [ ]:
# C05
data=u.load_data(BASE)
print('Dati SHA256:',data['sha256'])
print('N training, validation:',len(data['train_idx']),len(data['val_idx']))
print('Test valutato:',u.CONFIG['test_evaluated'])
assert len(data['train_idx'])==1077 and len(data['val_idx'])==360
for name,(file,url) in u.WEIGHTS.items():print(name,u.sha(BASE/'pesi'/file),url)
manifest=json.loads((BASE/"manifest.json").read_text())
assert data["sha256"]==manifest["data_sha256"], "Hash dati diverso dal manifest"
for _name,_info in manifest["weights"].items():
    assert u.sha(BASE/"pesi"/_info["file"])==_info["sha256"], f"Hash pesi inatteso: {_name}"
print("Manifest: dati e pesi verificati.")


<!-- C06 -->
## Input e dominio

Un **canale** è una griglia di valori. La replica RGB crea tre canali inizialmente uguali, senza aggiungere colore osservato. Il resize bilineare interpola valori su una griglia più fitta senza creare nuovi dettagli acquisiti. $u_c$ è l’intensità ridimensionata, $x_c=(u_c-\mu_c)/\sigma_c$ il valore normalizzato del canale c; media e deviazione standard sono le costanti ImageNet dichiarate.

Digits fornisce 8×8 intensità 0–16. Dividiamo per 16, replichiamo tre canali, interpola bilinearmente a 224×224, normalizziamo con μ=(.485, .456, .406), σ=(.229, .224, .225). L’ingrandimento non aggiunge dettagli. La replica produce canali RGB inizialmente uguali, ma la normalizzazione per canale li rende diversi. Questa pipeline comune adatta i checkpoint ImageNet al nostro dominio, senza riprodurre il crop di validazione ImageNet.

In [ ]:
# C07
sample=data['images'][data['train_idx'][:2]]
z=u.preprocess(sample)
print('Input:',sample.shape,'Input rete:',z.shape,'dtype:',z.dtype)
print('Pixel nero normalizzato:',(-u.MEAN/u.STD).flatten().tolist())
assert z.shape==(2,3,224,224)

<!-- C08 -->
## Checkpoint e testa

**CNN** significa *Convolutional Neural Network*, rete convoluzionale; ResNet18 usa filtri locali e blocchi residuali. **ViT** significa *Vision Transformer*: usa token, vettori che rappresentano patch dell’immagine, ed attenzione. **DeiT** significa *Data-efficient image Transformer*. Il token **CLS** è il vettore riassuntivo letto dal classificatore, non una classe aggiuntiva. La **Layer Normalization (LN)** normalizza le componenti di ciascun token senza statistiche mobili BN. Le teste di attenzione interne sono diverse dalla testa finale di classificazione. `strict=True` controlla le chiavi del checkpoint; SHA256 ne verifica l’identità.

ResNet18 ImageNet-1K V1, DeiT Tiny patch16_224 non distillato. Caricamento `strict=True` della testa ImageNet originale, poi sostituzione con una testa a 10 classi. Le dimensioni delle feature sono 512 e192. Stesso seed non significa stessi tensori tra architetture diverse.

In [ ]:
# C09
models={};dims={};results={}
for name in ['resnet18','deit_tiny']:
    models[name],dims[name]=u.make_model(name,BASE)
    print(name,'feature',dims[name],'parametri totali',sum(p.numel() for p in models[name].parameters()))
    results[name]={'feature_dim':dims[name],'total_parameters':sum(p.numel() for p in models[name].parameters()),'weight_file':u.WEIGHTS[name][0],'weight_sha256':u.sha(BASE/'pesi'/u.WEIGHTS[name][0])}

<!-- C10 -->
## N01 · Quali parametri apprendono?

**Autograd** costruisce il grafo delle operazioni per applicare la regola della catena nel backward. `requires_grad` controlla i gradienti dei parametri; i gruppi dell’ottimizzatore controllano chi riceve update; `eval()` sceglie la modalità dei moduli; `no_grad()` disabilita la registrazione del grafo. **BN**, Batch Normalization, ha parametri affini $\gamma,\beta$ e **buffer**, tensori di stato come media e varianza mobili. Nel protocollo i buffer BN restano fissi in eval; nel FT gli affini di `layer4` sono trainabili. `eval()` consente comunque le derivate.

**Completare C11.** Prima congela tutti i parametri, poi attiva la testa. Nel fine-tuning attiva anche `layer4` in ResNet18 e `blocks[-1]` più `norm` in DeiT. Mantieni il modello in `eval()` durante entrambi gli stadi. Motiva perché questo non blocca il gradiente. Quale parte di BatchNorm resta fissa?

**Risposta N01 (3–5 righe o piccola tabella):** [scrivere qui; citare gli output senza ricopiarli]

In [ ]:
# C11 · TODO
# Configurare i parametri trainabili e la modalità eval. Definire configure_stage e assegnarlo a u.configure_stage.
raise NotImplementedError("Completa il checkpoint C11")

In [ ]:
# C12
for name,m in models.items():
    for stage in ['lp','ft']:
        names=configure_stage(m,name,stage)
        print(name,stage,'parametri trainabili',sum(p.numel() for p in m.parameters() if p.requires_grad),'tensori',len(names))
        assert not m.training
    configure_stage(m,name,'lp')

<!-- C13 -->
## Linear probe

Una **cache** memorizza feature già calcolate per riusarle; è valida con backbone, buffer, input e preprocessing fissi. Con B immagini e D feature, $H$ ha forma $B\times D$, $W$ forma $10\times D$, b dieci elementi e gli score $S=HW^T+b$ forma $B\times10$. Il bias è condiviso fra le righe; i parametri della testa sono $10D+10$. Il codice chiama `logits` gli score; una variabile `z` eventualmente usata per un input preprocessato non cambia il significato matematico di s/S.

**Esempio analitico a due classi**, distinto dalle dieci classi Digits: h=(1, 2), W=[[1, -1], [-1, 1]], b=(0, 0) danno s=(-1, 1). Softmax dà circa (0.119203, 0.880797); con target 0, CE=$-\log p_0\approx2.126928$. In LP h rimane fissa mentre la testa apprende.

C14 e C15 estraggono una sola volta le feature training/validation in eval, poi apprendono soltanto la testa con Adam lr=.001 per 20 epoche. Nessun gradiente o cambio di buffer nel backbone. La cache è valida solo finché backbone e preprocessing restano identici. Il costo LP include la prima estrazione delle feature, non solo il fit della testa.

### P01 · Dalla rappresentazione alla testa LP

La testa finale riceve una feature di dimensione D, come la testa del modello della lezione 13. Per un batch B e **10 classi**, scrivere le forme di H, W, b e S. Contare pesi e bias della testa per **D=512 (ResNet18)** e **D=192 (DeiT Tiny)**. Un token CLS aggiunge una classe d'uscita? Motivare senza confondere la testa finale con le teste di attenzione interne. Usare le dimensioni già stampate da C09: non occorre eseguire un altro modello o modificare il fit.

**Risposta P01 (3–5 righe o piccola tabella):** [scrivere qui]

In [ ]:
# C14
results['resnet18']['lp']=u.run_lp('resnet18',models['resnet18'],data,OUT)
print({k:v for k,v in results['resnet18']['lp'].items() if k!='history' and k!='validation'})
print('Accuracy validation:',results['resnet18']['lp']['validation']['accuracy'])

In [ ]:
# C15
results['deit_tiny']['lp']=u.run_lp('deit_tiny',models['deit_tiny'],data,OUT)
print({k:v for k,v in results['deit_tiny']['lp'].items() if k!='history' and k!='validation'})
print('Accuracy validation:',results['deit_tiny']['lp']['validation']['accuracy'])

In [ ]:
# C16
assert results['resnet18']['lp']['order_sha256']==results['deit_tiny']['lp']['order_sha256']
for name,r in results.items():print(name,'LP checkpoint',r['lp']['selected_epoch'],'accuracy',r['lp']['validation']['accuracy'],'CE',r['lp']['validation']['ce'])

<!-- C17 -->
## Fine-tuning parziale

Il **prefisso** è la parte iniziale della rete, il **suffisso** quella finale. Nel FT parziale il gradiente attraversa soltanto la parte trainabile; il forward va ricalcolato perché il mapping delle feature può cambiare. Nel toy di C13, $g_s=p-e_0\approx(-0.880797, 0.880797)$, $g_W=g_s h^T$, $g_b=g_s$ e $g_h=W^Tg_s\approx(-1.761594, 1.761594)$. In LP questo ultimo segnale non aggiorna il backbone; in FT può attraversare il suffisso sbloccato. Il gradiente è una derivata: l’update viene eseguito dallo step di Adam.

Ogni modello prosegue dalla propria migliore testa LP. Nuovo Adam, senza ereditare momenti LP. Head lr=.001, backbone lr=.0001, weight_decay=0. ResNet sblocca l’ultimo stage residuo, DeiT l’ultimo blocco e LayerNorm finale. Non sono quantità uguali di parametri. La cache LP non è garantita per il nuovo stato della funzione estrattrice: va ricalcolata o verificata, senza presumere che ogni input cambi numericamente. La BN usa sempre statistiche ImageNet fisse, mentre gli affine γ/β di layer4 possono apprendere.

### P02 · Residuo e validità della cache

Riprendere il **caso residuale analitico senza ReLU** della lezione 13, distinto dall'intero ResNet18 o DeiT: x=(1, 2) e h=x+αx. Calcolare h con **α=−0.2**, poi dopo una variazione ipotetica del suffisso a **α=−0.1**. La vecchia cache rappresenta il secondo stato?

Classificare poi i cambiamenti seguenti, mantenendo fissi tutti gli altri elementi: **sola testa finale**, **pesi del suffisso**, **preprocessing**, **buffer BN pertinenti all'estrattore**. Quando resta garantito il riuso delle feature e quando occorre ricalcolare o verificarne la validità? Non eseguire training né un ulteriore forward dei modelli: bastano il calcolo del toy e il ragionamento sul contratto della cache.

**Risposta P02 (3–5 righe o piccola tabella):** [scrivere qui]

<!-- C18 -->
## N02 · Gruppi di parametri
**Completare C19.** Costruisci due gruppi senza sovrapposizioni, includendo esattamente i parametri trainabili. Restituisci learning rate separati. Spiega perché una nuova chiamata ad Adam costituisce una nuova fase di ottimizzazione e perché `requires_grad=False` da solo non congela i buffer BN.

**Risposta N02 (3–5 righe o piccola tabella):** [scrivere qui; citare gli output senza ricopiarli]

In [ ]:
# C19 · TODO
# Costruire i gruppi Adam testa/backbone. Definire optimizer_groups e assegnarlo a u.optimizer_groups.
raise NotImplementedError("Completa il checkpoint C19")

In [ ]:
# C20
results['resnet18']['ft']=u.run_ft('resnet18',models['resnet18'],data,OUT)
print('Checkpoint FT ResNet18:',results['resnet18']['ft']['selected_epoch'])

In [ ]:
# C21
results['deit_tiny']['ft']=u.run_ft('deit_tiny',models['deit_tiny'],data,OUT)
print('Checkpoint FT DeiT Tiny:',results['deit_tiny']['ft']['selected_epoch'])

In [ ]:
# C22
assert results['resnet18']['ft']['order_sha256']==results['deit_tiny']['ft']['order_sha256']
for name,r in results.items():
    print(name,'LP',r['lp']['validation']['accuracy'],'FT',r['ft']['validation']['accuracy'],'delta pp',100*(r['ft']['validation']['accuracy']-r['lp']['validation']['accuracy']))
    assert r['ft']['frozen_tensors_unchanged'] and r['ft']['bn_buffers_unchanged']

<!-- C23 -->
## N03 · Interpretazione del confronto
Un aumento di accuracy validation è una misura sul protocollo realizzato. Il checkpoint è selezionato sulla stessa validation. Non prova che questa architettura sia superiore, né garantisce un miglioramento su un test indipendente. Identifica tre differenze ancora presenti tra i modelli e una procedura per una futura stima più stabile.

**Risposta N03 (3–5 righe o piccola tabella):** [scrivere qui; citare gli output senza ricopiarli]

<!-- C24 -->
## Costo definito prima della misura

**FP32** usa 32 bit, cioè 4 byte per parametro. P è il numero totale di parametri e un **MiB** vale $2^{20}=1048576$ byte; $4P/2^{20}$ descrive i soli pesi, senza gradienti, momenti Adam, attivazioni e buffer. La **latenza** è il tempo di una richiesta; il **throughput** è il numero di esempi al secondo con batch dichiarato. Il **warm-up** consiste in esecuzioni preliminari escluse dalla misura. La mediana è il valore centrale delle osservazioni ordinate; IQR, intervallo interquartile, è $Q_3-Q_1$. I tempi di riferimento sono locali su CPU; non sono misure di Colab.

Parametri totali e trainabili sono conteggi, non memoria di picco. La memoria teorica dei soli parametri FP32 vale4P byte. Tempo LP = cache training+validation e fit/testa. Tempo FT = training e valutazioni a fine epoca. Latenza batch 1 = preprocessing+forward CPU, dopo3 warm-up, 7 ripetizioni, mediana e intervallo interquartile. Esclude caricamento file/pesi. Il timer CPU è sincrono.

In [ ]:
# C25
for name,m in models.items():
    results[name]['latency']=u.measure_latency(m,data['images'][data['val_idx']])
    results[name]['parameter_MiB_fp32']=4*results[name]['total_parameters']/2**20
    print(name,'parametri FP32 MiB',results[name]['parameter_MiB_fp32'],'latenza',results[name]['latency'])

<!-- C26 -->
## N04 · Budget e confronto
Stesse epoche e stessi ID implicano stesso numero di esempi e aggiornamenti per stadio. Non implicano stessa durata, FLOP o memoria. Una cache già disponibile va dichiarata e il suo costo iniziale separato. Latenza batch 1 e throughput batch 32 rispondono a domande diverse. Non misuriamo la memoria di picco in questo notebook.

**Risposta N04 (3–5 righe o piccola tabella):** [scrivere qui; citare gli output senza ricopiarli]

<!-- C27 -->
## Stress prefissato

Uno **stress test** valuta checkpoint fissi su alterazioni prefissate del target. **OOD** significa *out-of-distribution*, fuori distribuzione: lo stress su cifre note non è una prova su classi semantiche sconosciute. La sigla **pp** indica punti percentuali: si sottraggono due accuracy espresse in percentuale, oppure si moltiplica per 100 la differenza fra le loro frazioni. Il calo relativo usa invece il valore pulito come denominatore.

Su tutti i 360 ID validation applica (a) traslazione verso destra di 1 pixel originario con riempimento nero, (b) intensità×.5, prima del resize. Le etichette vengono mantenute come ipotesi da controllare visivamente: il bordo potrebbe perdere un tratto. Sono due alterazioni di acquisizione sintetiche, non nuovi oggetti o classi sconosciute. Nessuna scelta del checkpoint usa queste metriche.

<!-- C28 -->
## N05 · Delta e unità
**Completare C29.** Calcola la differenza di accuracy in **punti percentuali**, perturbata meno pulita. Per esempio .80 contro .90 produce−10pp, non−.10pp. Il calo relativo avrebbe un altro denominatore. Come gestiresti una trasformazione che cambia il significato di una cifra?

**Risposta N05 (3–5 righe o piccola tabella):** [scrivere qui; citare gli output senza ricopiarli]

In [ ]:
# C29 · TODO
# Definire delta_pp(perturbed_accuracy,clean_accuracy) e verificare il caso .8,.9.
raise NotImplementedError("Completa il checkpoint C29")

In [ ]:
# C30
for name,m in models.items():
    results[name]['stress']=u.stress_result(m,data)
    for kind in u.CONFIG['corruptions']:
        a=results[name]['stress'];assert abs(delta_pp(a[kind]['accuracy'],a['clean']['accuracy'])-a[kind]['delta_accuracy_pp'])<1e-9
print('Stress completato sui checkpoint FT, senza ulteriore adattamento.')

In [ ]:
# C31
for name,r in results.items():
    for kind,v in r['stress'].items():print(name,kind,v['correct'],'/',v['n'],'accuracy',v['accuracy'],'delta pp',v.get('delta_accuracy_pp',0))

In [ ]:
# C32
import matplotlib.pyplot as plt
ids=data['val_idx'][:6];fig,axes=plt.subplots(3,6,figsize=(12,6))
for row,kind in enumerate(['clean']+u.CONFIG['corruptions']):
    x=data['images'][ids];x=x if kind=='clean' else u.corrupt(x,kind)
    for j,idx in enumerate(ids):
        axes[row,j].imshow(x[j],cmap='gray',vmin=0,vmax=1,interpolation='nearest');axes[row,j].set_title(f'ID {int(idx)} · y={int(data["target"][idx])}');axes[row,j].set_xticks([]);axes[row,j].set_yticks([])
    axes[row,0].set_ylabel(['Pulita','Spostata +1px','Contrasto ×0.5'][row])
fig.tight_layout();fig.savefig(OUT/'stress_examples.png',dpi=180,bbox_inches='tight');plt.show()

<!-- C33 -->
## N06 · Lettura guidata: diagnosi della modalità

**Formato del ripristino C35.** `ft_delta` conserva i valori finali dei tensori adattati. Si carica la medesima base verificata e si sostituiscono i tensori per nome: non si sommano differenze aritmetiche. Il ripristino dell’inferenza è distinto dalla ripresa di Adam, che richiede anche stato dell’ottimizzatore, generatori e punto del ciclo.

Un collega congela tutti i parametri ma chiama `model.train()` sul backbone ResNet. Il risultato è un linear probe con rappresentazione fissa? Leggere il codice già fornito in C34 e formulare una diagnosi breve; non occorre implementare un altro esperimento. La prova usa una copia, senza alterare i modelli confrontati.

C34 è una dimostrazione già pronta su copia: è sufficiente leggere gli output di riferimento se si lavora in recupero. Ulteriori varianti o repliche sono facoltative e non valutate.


**Risposta N06 (3–5 righe o piccola tabella):** [scrivere qui; citare gli output senza ricopiarli]

In [ ]:
# C34
probe=copy.deepcopy(models['resnet18'])
for p in probe.parameters():p.requires_grad_(False)
probe.train();before={k:v.clone() for k,v in probe.named_buffers()}
with torch.no_grad():probe(u.preprocess(data['images'][data['train_idx'][:32]]))
changed=[k for k,v in probe.named_buffers() if not torch.equal(v,before[k])]
print('Buffer modificati:',len(changed),changed[:6])
assert len(changed)>0
del probe

In [ ]:
# C35
# Ricostruisce ogni FT da pesi originali + delta e verifica l'identità dell'output.
for name,m in models.items():
    restored,_=u.make_model(name,BASE);delta=torch.load(OUT/f'ft_delta_{name}.pt',weights_only=True);state=restored.state_dict();state.update(delta);restored.load_state_dict(state,strict=True);restored.eval()
    with torch.no_grad():
        x=u.preprocess(data['images'][data['val_idx'][:8]]);err=float((m(x)-restored(x)).abs().max())
    print(name,'errore massimo ricarica',err);assert err==0
    results[name]['reload_identical']=True

<!-- C36 -->
## Model comparison card · Sintesi individuale F01–F05

Scrivere qui la sintesi **entro 500 parole complessive**, . Non serve una relazione separata: richiamare celle, tabelle e figure già presenti. P01/P02 e N01–N06 restano nei rispettivi blocchi, senza ripetere gli svolgimenti. Il report collega conclusioni a protocollo, checkpoint, unità e limiti; un delta di pesi non è un delta di accuracy.

**F01 — Protocollo e provenienza:** [dati/split/hash, checkpoint pretrained, preprocessing, ambiente, seed, stadi/budget e regola di selezione]

**F02 — Prestazioni:** [citare le quattro righe LP/FT di C16/C22/C37 e i delta; distinguere validation selezionata da test indipendente]

**F03 — Costo:** [citare cache+fit LP, FT incrementale, pipeline totale, parametri e latenza con hardware, unità e confini]

**F04 — Stress:** [trasformazioni prefissate, delta in punti percentuali, esempi e validità delle etichette; richiamare C30–C32]

**F05 — Decisione e limite:** [requisito dichiarato, conclusione condizionata e un controllo o futuro esperimento proposto, senza eseguirlo]

Prima di salvare, controllare C11/C19/C29 completate, risposte P01/P02 e N01–N06, output e sintesi. C37 esporta JSON/CSV; scaricare anche notebook e cartella OUT compressa. Conservare hash e provenienza dei pesi originali. Il test resta escluso; non è richiesto un risultato favorevole né una soglia minima di accuracy. Un fit non eseguito o un risultato mancante va dichiarato; le misure di recupero restano attribuite all’archivio originale.

In [ ]:
# C37
rows=u.export_comparison(results,OUT)
record={'environment':ENV,'protocol':u.CONFIG,'data_sha256':data['sha256'],'models':results,'test_evaluated':False}
(OUT/'metriche_finali.json').write_text(json.dumps(record,indent=2))
print('Esportati:',OUT/'confronto.csv',OUT/'metriche_finali.json')
print(rows)

<!-- C38 -->
## Recupero e ripartenze
Per leggere senza eseguire: apri `recupero/Risultati_riferimento.html`. Se salti il fit, usa il notebook `recupero/Ripristino_risultati.ipynb`, che carica pesi originali+delta e i risultati misurati. Non mescola queste misure con una tua nuova corsa. Per rieseguire solo FT, esegui C03, C05, C09, C11, C19 e carica **la testa LP** prima di C20–C21. Una nuova testa casuale cambierebbe il protocollo. Per una corsa completa riavvia il kernel e procedi dall’inizio.

Per la ripresa del solo FT, prima di C20 importare anche il riferimento LP necessario a C22, senza cambiare i risultati del nuovo FT:

```python
ref = json.loads((BASE / "recupero/metriche_finali.json").read_text())
for name, model in models.items():
    head_of(model, name).load_state_dict(torch.load(
        BASE / "recupero" / f"lp_head_{name}.pt", weights_only=True))
    results[name]["lp"] = copy.deepcopy(ref["models"][name]["lp"])
    results[name]["lp"]["origin"] = "LP di riferimento importato"
    results[name]["lp"]["source_environment"] = ref["environment"]
    model.eval()
```

Le misure LP restano di riferimento; quelle FT saranno della nuova corsa. Non attribuire i tempi importati al computer corrente.


Il recupero non sostituisce gli esercizi individuali: completare C11/C19/C29 e rispondere a P01/P02 e N01–N06. Dichiarare nella sintesi C36 le parti non eseguite personalmente; non attribuire al proprio computer tempi o metriche importati. Non occorre ripetere il training soltanto per rispondere ai richiami testuali.

<!-- C39 -->
## Fonti
PyTorch torchvision ResNet18 IMAGENET1K_V1: https://docs.pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html

DeiT, Touvron et al., ICML2021: https://proceedings.mlr.press/v139/touvron21a.html

Pesi e architettura originali: https://github.com/facebookresearch/deit

Transfer learning tutorial PyTorch: https://docs.pytorch.org/tutorials/beginner/transfer_learning_tutorial.html

Digits: https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_digits.html